In [ ]:
!pip install -q transformers==4.45.0 peft==0.13.0 trl==0.11.0 \
             datasets==2.19.0 accelerate==0.34.0 sentencepiece scipy lm-eval

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 72.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 103.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch, json, random
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets
from lm_eval.models.huggingface import HFLM

MODEL_ID    = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
OUTPUT_DIR  = "/kaggle/working/tinyllama-medical-lora"
MERGED_DIR  = "/kaggle/working/tinyllama-medical-merged"
MAX_SEQ_LEN = 768
BATCH_SIZE  = 2
GRAD_ACCUM  = 16
EPOCHS     = 3
LR         = 1e-4
LORA_R     = 64
LORA_ALPHA = 256
LORA_DROPOUT= 0.05

print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules = ["q_proj", "k_proj", "v_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

trainable params: 12,255,232 || all params: 1,112,303,616 || trainable%: 1.1018


In [4]:
def fmt(text):
    return str(text).strip()

def format_casehold(ex):

    choices = "\n".join([
        f"({i}) {fmt(choice)}"
        for i, choice in enumerate(ex["endings"])
    ])

    prompt = (
        f"Legal Context:\n{fmt(ex['context'])}\n\n"
        f"Possible Holdings:\n{choices}\n\n"
        f"Answer: ("
    )

    answer = f"{ex['label']})"

    return {
        "prompt": prompt,
        "answer": answer,
        "text": prompt + answer
    }


# Load dataset
casehold = load_dataset("lex_glue", "case_hold")

casehold_train = casehold["train"]
casehold_val   = casehold["validation"]


# OPTIONAL: smaller subset for faster training
TRAIN_SAMPLES = 5000
VAL_SAMPLES   = 1000

casehold_train = (
    casehold_train
    .shuffle(seed=42)
    .select(range(TRAIN_SAMPLES))
)

casehold_val = (
    casehold_val
    .shuffle(seed=42)
    .select(range(VAL_SAMPLES))
)


# Format dataset
casehold_train = casehold_train.map(
    format_casehold,
    remove_columns=casehold_train.column_names
)

casehold_val = casehold_val.map(
    format_casehold,
    remove_columns=casehold_val.column_names
)


print(f"Train: {len(casehold_train):,}")
print(f"Val  : {len(casehold_val):,}")

print("\nSample:\n")
print(casehold_train[0]["text"])

Generating train split:   0%|          | 0/45000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3900 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 5,000
Val  : 1,000

Sample:

Legal Context:
that could not be reconstructed and was, therefore, so deficient as to have made a fair and complete direct appeal of Perryman’s convictions an impossibility. The portions of the record cited to by Perryman at the hearing on his motion to reconstruct the record contain such statements as “Sidebar; inaudible to report.” See 2006 Transcript at 197, 200, 204, 207, 214, 225, 238. The transcript on the direct appeal from the 2006 trial included these notations indicating that certain portions were inaudible. Perryman did not raise the claims of an insufficient record on direct appeal and does not allege that his trial counsel or appellate counsel were ineffective on these bases. Consequently, we conclude that Perryman waived these claims. See Reed v. State, 866 N.E.2d 767, 768 (Ind.2007) (<HOLDING>); Sanders v. State, 765 N.E.2d 591, 592

Possible Holdings:
(0) holding that issues not properly presented to the bankruptcy court  cannot be ra

In [ ]:
from transformers import Trainer

def make_collator(tokenizer, max_len):
    def collate(batch):

        full_texts = [b["text"] for b in batch]

        enc = tokenizer(
            full_texts,
            truncation=True,
            max_length=max_len,
            padding="max_length",
            return_tensors="pt"
        )

        labels = enc["input_ids"].clone()

        for i, text in enumerate(full_texts):

            split_marker = "Answer: ("

            prompt_part = (
                text[:text.rfind(split_marker) + len(split_marker)]
            )

            prompt_ids = tokenizer(
                prompt_part,
                add_special_tokens=False
            )["input_ids"]

            prompt_len = len(prompt_ids)

            # Ignore prompt tokens in loss
            labels[i, :prompt_len] = -100

        # Ignore padding
        labels[labels == tokenizer.pad_token_id] = -100

        enc["labels"] = labels

        return enc

    return collate


training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,

    lr_scheduler_type="cosine",

    warmup_ratio=0.05,

    fp16=False,
    bf16=True,

    logging_steps=50,

    # Better for long legal sequences
    eval_strategy="epoch",
    save_strategy="no",

    report_to="none",

    group_by_length=False,

    optim="adamw_torch",

    weight_decay=0.01,

    remove_unused_columns=False,

    load_best_model_at_end=False,
)


trainer = Trainer(

    model=model,

    tokenizer=tokenizer,

    data_collator=make_collator(
        tokenizer,
        MAX_SEQ_LEN
    ),

    train_dataset=casehold_train,
    eval_dataset=casehold_val,

    args=training_args,
)


trainer.train()

print("Done!")

Epoch,Training Loss,Validation Loss
0,0.272000,0.229515
1,0.179200,0.193360


In [8]:
from lm_eval.models.huggingface import HFLM
import lm_eval
import json


lm_model_ft = HFLM(
    pretrained=trainer.model,
    tokenizer=tokenizer,
    batch_size=8,
)


results_after = lm_eval.simple_evaluate(
    model=lm_model_ft,
    tasks=["lexglue_casehold"],
)


# Print full metrics
print(json.dumps(results_after["results"], indent=2))


# Main accuracy metric
casehold_acc = (
    results_after["results"]["casehold"]["acc,none"] * 100
)

print(f"\nCaseHOLD Accuracy : {casehold_acc:.2f}")

KeyError: 'lexglue_casehold'

In [ ]:
print(results_after)

In [ ]:
from datasets import load_dataset

casehold_test = load_dataset(
    "lex_glue",
    "case_hold",
    split="test"
)

# Optional smaller subset for quick testing
casehold_test = (
    casehold_test
    .shuffle(seed=42)
    .select(range(1000))
)

print(casehold_test[0])

In [ ]:
import torch
from tqdm import tqdm

model.eval()

correct = 0
total = 0

for ex in tqdm(casehold_test):

    choices = ex["endings"]

    prompt = (
        f"Legal Context:\n{ex['context']}\n\n"
        f"Possible Holdings:\n"
    )

    for i, c in enumerate(choices):
        prompt += f"({i}) {c}\n"

    prompt += "\nAnswer: ("

    scores = []

    for i in range(5):

        candidate = f"{i})"

        full_text = prompt + candidate

        inputs = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[:, :-1]
        labels = inputs["input_ids"][:, 1:]

        loss_fct = torch.nn.CrossEntropyLoss(
            reduction="mean"
        )

        loss = loss_fct(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )

        scores.append(-loss.item())

    pred = scores.index(max(scores))

    if pred == ex["label"]:
        correct += 1

    total += 1


accuracy = 100 * correct / total

print(f"\nCaseHOLD TEST Accuracy: {accuracy:.2f}%")

In [1]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    torch_dtype=torch.float16,
    device_map="auto"
)

NameError: name 'torch' is not defined